In [0]:
from pyspark.sql import functions as F

from pyspark.sql.window import Window

# ------------------------------------------------------------

# CONFIG

# ------------------------------------------------------------

SILVER_TICKETS = "workspace.slainte_silver.easyvista_tickets_silver_demo"

DIM_EMPLOYEE   = "workspace.slainte_gold.dim_employee"

DIM_TEAM       = "workspace.slainte_gold.dim_team"

DIM_PRIORITY   = "workspace.slainte_gold.dim_priority"

DIM_STATUS     = "workspace.slainte_gold.dim_status"

DIM_TT         = "workspace.slainte_gold.dim_ticket_type"

DIM_SLA        = "workspace.slainte_gold.dim_sla"

# NEW DIMS

DIM_SOURCE     = "workspace.slainte_gold.dim_source"

DIM_CLIENT     = "workspace.slainte_gold.dim_client"

DIM_PROJECT    = "workspace.slainte_gold.dim_project"

GOLD_DB        = "workspace.slainte_gold"

FACT_TABLE     = f"{GOLD_DB}.fact_ticket_performance"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

# ------------------------------------------------------------

# 1️⃣ Read SILVER tickets (RAW)

# ------------------------------------------------------------

b_raw = (

    spark.table(SILVER_TICKETS)

    .select(

        F.col("ticket_number"),

        F.col("entity"),

        F.col("ticket_type"),

        F.col("priority").cast("int").alias("priority_code"),

        F.col("support_group"),

        F.col("assignee"),

        F.col("status"),

        F.col("created_at"),

        F.col("assigned_at"),

        F.col("last_update"),

        F.col("source"),

        F.col("source_user_id"),

        F.col("project"),

        F.col("ingestion_ts")

    )

)

# ------------------------------------------------------------

# 1️⃣.5 KEEP ONLY LATEST VERSION PER TICKET (IMPORTANT FIX)

# ------------------------------------------------------------

w = Window.partitionBy("ticket_number").orderBy(F.col("ingestion_ts").desc())

b = (

    b_raw

    .withColumn("rn", F.row_number().over(w))

    .filter(F.col("rn") == 1)

    .drop("rn")

    .withColumn("ticket_type_key", F.upper(F.trim(F.col("ticket_type"))))

    .withColumn("status_key", F.upper(F.trim(F.col("status"))))

    .alias("b")

)

# ------------------------------------------------------------

# 2️⃣ Read DIM tables

# ------------------------------------------------------------

e  = spark.table(DIM_EMPLOYEE).alias("e")

t  = spark.table(DIM_TEAM).alias("t")

pr = spark.table(DIM_PRIORITY).alias("pr")

st = spark.table(DIM_STATUS).alias("st")

tt = (

    spark.table(DIM_TT)

    .withColumn("ticket_type_key", F.upper(F.trim(F.col("ticket_type_name"))))

    .alias("tt")

)

s   = spark.table(DIM_SLA).alias("s")

src = spark.table(DIM_SOURCE).alias("src")

cl  = spark.table(DIM_CLIENT).alias("cl")

pj  = spark.table(DIM_PROJECT).alias("pj")

# ------------------------------------------------------------

# 3️⃣ Joins

# ------------------------------------------------------------

j = (

    b

    # EMPLOYEE

    .join(

        e,

        (b.source_user_id == e.source_user_id) &

        (b.project == e.project) &

        (b.assignee == e.employee_name),

        "left"

    )

    # TEAM

    .join(

        t,

        (b.source_user_id == t.source_user_id) &

        (b.project == t.project) &

        (b.support_group == t.team_name),

        "left"

    )

    # PRIORITY

    .join(

        pr,

        (b.source_user_id == pr.source_user_id) &

        (b.project == pr.project) &

        (b.priority_code == pr.priority_code),

        "left"

    )

    # STATUS

    .join(

        st,

        (b.source_user_id == st.source_user_id) &

        (b.project == st.project) &

        (b.status_key == st.status_code),

        "left"

    )

    # TICKET TYPE

    .join(

        tt,

        (b.source_user_id == tt.source_user_id) &

        (b.project == tt.project) &

        (b.ticket_type_key == tt.ticket_type_key),

        "left"

    )

    # SLA

    .join(

        s,

        (b.source_user_id == s.source_user_id) &

        (b.project == s.project) &

        (b.priority_code == s.priority_code),

        "left"

    )

    # SOURCE

    .join(

        src,

        b.source == src.source_name,

        "left"

    )

    # CLIENT

    .join(

        cl,

        b.source_user_id == cl.source_user_id,

        "left"

    )

    # PROJECT

    .join(

        pj,

        (b.source_user_id == pj.source_user_id) &

        (b.project == pj.project),

        "left"

    )

)

# ------------------------------------------------------------

# 4️⃣ FINAL FACT

# ------------------------------------------------------------

df_fact_final = j.select(

    b.ticket_number,

    # 🔑 CONTEXT (MANDATORY)

    b.source_user_id,

    b.project,

    # 🔑 IDS

    src.source_id,

    cl.client_id,

    pj.project_id,

    e.employee_id,

    t.team_id,

    tt.ticket_type_id,

    pr.priority_id,

    st.status_id,

    s.sla_id,

    # ⏱ timestamps

    b.created_at,

    b.assigned_at,

    b.last_update,

    b.ingestion_ts

)
 

# ------------------------------------------------------------

# 5️⃣ Write FACT

# ------------------------------------------------------------

df_fact_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(FACT_TABLE)

print("✅ fact_ticket_performance UPDATED (latest ticket version only)")

display(

    spark.table(FACT_TABLE)

    .orderBy("client_id", "project_id", "ticket_number")

)
 

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.slainte_gold.v_ticket_fact_dim AS

SELECT

    -- ======================

    -- Ticket identifiers

    -- ======================

    f.ticket_number,

    f.source_user_id          AS client,

    f.project,

    -- ======================

    -- Source

    -- ======================

    src.source_name           AS source,

    -- ======================

    -- Business dimensions

    -- ======================

    tt.ticket_type_name       AS ticket_type,

    pr.priority_code          AS priority,

    st.status_label           AS status,

    tm.team_name              AS team,

    emp.employee_name         AS assignee,

    -- ======================

    -- Dates

    -- ======================

    f.created_at,

    f.assigned_at,

    f.last_update,

    f.ingestion_ts

FROM workspace.slainte_gold.fact_ticket_performance f

-- SOURCE

LEFT JOIN workspace.slainte_gold.dim_source src

    ON f.source_id = src.source_id

-- PROJECT (scoped correctly)

LEFT JOIN workspace.slainte_gold.dim_project pj

    ON  f.project_id     = pj.project_id

    AND f.source_user_id = pj.source_user_id

    AND f.project        = pj.project

-- EMPLOYEE

LEFT JOIN workspace.slainte_gold.dim_employee emp

    ON  f.employee_id    = emp.employee_id

    AND f.source_user_id = emp.source_user_id

    AND f.project        = emp.project

-- TEAM

LEFT JOIN workspace.slainte_gold.dim_team tm

    ON  f.team_id        = tm.team_id

    AND f.source_user_id = tm.source_user_id

    AND f.project        = tm.project

-- PRIORITY (business value only)

LEFT JOIN workspace.slainte_gold.dim_priority pr

    ON  f.priority_id    = pr.priority_id

    AND f.source_user_id = pr.source_user_id

    AND f.project        = pr.project

-- STATUS

LEFT JOIN workspace.slainte_gold.dim_status st

    ON  f.status_id      = st.status_id

    AND f.source_user_id = st.source_user_id

    AND f.project        = st.project

-- TICKET TYPE

LEFT JOIN workspace.slainte_gold.dim_ticket_type tt

    ON  f.ticket_type_id = tt.ticket_type_id

    AND f.source_user_id = tt.source_user_id

    AND f.project        = tt.project

;
 

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.slainte_gold.v_ticket_dashboard_final AS

SELECT

    -- ======================

    -- Ticket core

    -- ======================

    f.ticket_number,

    f.client,

    f.project,

    f.source,

    -- ======================

    -- Business dimensions

    -- ======================

    f.ticket_type,

    f.priority,

    f.status,

    f.team,

    f.assignee,

    -- ======================

    -- SLA results

    -- ======================

    sla.response_breach_flag,

    sla.resolution_breach_flag,

    sla.sla_breached_flag,

    -- Optional elapsed times

    sla.response_minutes_elapsed,

    sla.resolution_hours_elapsed,

    -- ======================

    -- Penalty results

    -- ======================

    pen.penalty_code,

    pen.penalty_amount,

    pen.penalty_unit,

    -- ======================

    -- Dates

    -- ======================

    f.created_at,

    f.assigned_at,

    f.last_update,

    f.ingestion_ts

FROM workspace.slainte_gold.v_ticket_fact_dim f

-- ======================

-- SLA (business-grain join)

-- ======================

LEFT JOIN workspace.slainte_silver.v_ticket_sla_final sla

    ON  f.ticket_number  = sla.ticket_number

    AND f.client         = sla.source_user_id

    AND f.project        = sla.project

-- ======================

-- Penalty (business-grain join)

-- ======================

LEFT JOIN workspace.slainte_silver.v_ticket_penalty_final pen

    ON  f.ticket_number  = pen.ticket_number

    AND f.client         = pen.source_user_id

    AND f.project        = pen.project

;
 